## The objective of this notebook is to transform raw data into a cleaned version 

In [66]:
# Importing libraries

import pandas as pd
import numpy as np
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler,Normalizer
from sklearn.model_selection import train_test_split

In [67]:
# Reading raw data

liqdb_data = pd.read_csv("raw_data/LiqDB read data.csv")

F:\Python\lib\site-packages\IPython\core\interactiveshell.py:3457: DtypeWarning: Columns (12,13,16) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [68]:
liqdb_data.head(10)

,ID,SRP,PRJ,Organism,Sample,Runs,Instrument,Release,Sex,Fluid,...,hsa-miR-3658,hsa-miR-3620-5p,hsv1-miR-H4-5p,hsa-miR-4798-3p,hsa-miR-5702,hsa-miR-548ao-5p,hsa-miR-4790-5p,hsa-miR-4684-5p,hsa-miR-4539,hsa-miR-10525-3p
0,0,SRP020486,PRJNA196121,Homo sapiens,SAMN01997249,"SRR804662,SRR804663",Illumina HiSeq 2000,16-05-2013,NaN,plasma,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,SRP020486,PRJNA196121,Homo sapiens,SAMN01997250,"SRR804664,SRR804665,SRR804666",Illumina HiSeq 2000,16-05-2013,NaN,plasma,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,SRP020486,PRJNA196121,Homo sapiens,SAMN01997251,"SRR804667,SRR804668",Illumina HiSeq 2000,16-05-2013,NaN,plasma,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,SRP020486,PRJNA196121,Homo sapiens,SAMN01997252,"SRR804669,SRR804670,SRR804671",Illumina HiSeq 2000,16-05-2013,NaN,plasma,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,SRP020486,PRJNA196121,Homo sapiens,SAMN01997253,"SRR804672,SRR804673",Illumina HiSeq 2000,16-05-2013,NaN,plasma,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,5,SRP020486,PRJNA196121,Homo sapiens,SAMN01997254,"SRR804674,SRR804675",Illumina HiSeq 2000,16-05-2013,NaN,plasma,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,6,SRP020486,PRJNA196121,Homo sapiens,SAMN01997255,"SRR804676,SRR804677",Illumina HiSeq 2000,16-05-2013,NaN,plasma,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,7,SRP020486,PRJNA196121,Homo sapiens,SAMN01997256,"SRR804678,SRR804679",Illumina HiSeq 2000,16-05-2013,NaN,plasma,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,8,SRP020486,PRJNA196121,Homo sapiens,SAMN01997257,"SRR804680,SRR804681",Illumina HiSeq 2000,16-05-2013,NaN,plasma,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,9,SRP020486,PRJNA196121,Homo sapiens,SAMN01997258,"SRR804682,SRR804683,SRR804684",Illumina HiSeq 2000,16-05-2013,NaN,plasma,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [69]:
# Examining the datatype of columns

liqdb_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1606 entries, 0 to 1605
Columns: 2441 entries, ID to hsa-miR-10525-3p
dtypes: bool(1), float64(2420), int64(4), object(16)
memory usage: 29.9+ MB


In [70]:
# Dropping columns which indicate record metadata and is not needed for prediction

liqdb_data_cleaned_v1 = (
    liqdb_data
    .drop(columns = ["ID","SRP","PRJ","Organism","Sample","Runs","Instrument","Release","Extraction","Library",
                      "Experiment","Age","Sex","RC","Exosome","Description","Fluid"],axis=1)
)

In [71]:
# Examining the data quality 

cols_with_nulls = []
data_columns = liqdb_data_cleaned_v1.columns
for col in data_columns:
    if liqdb_data_cleaned_v1[col].isnull().sum() != 0:
        cols_with_nulls.append(col)
        
cols_with_nulls

['Healthy', 'Cancer']

In [72]:
# Exploring the unique values in the Healthy and Cancer columns

liqdb_data_cleaned_v1["Healthy"].unique()

array([True, False, nan], dtype=object)

In [73]:
liqdb_data_cleaned_v1["Cancer"].unique()

array([False, True, nan], dtype=object)

In [74]:
# Let us reconcile these two columns which will become our target variable
# Removing rows where neither column value is True

liqdb_data_cleaned_v1 = (
    liqdb_data_cleaned_v1[~((liqdb_data_cleaned_v1["Healthy"] != True) & (liqdb_data_cleaned_v1["Cancer"] != True))]
)

len(liqdb_data_cleaned_v1)

1495

In [75]:
# We can now treat the Healthy column as the target variable

liqdb_data_cleaned_v1[["Healthy","Cancer"]].isnull().sum()

Healthy     0
Cancer     28
dtype: int64

In [76]:
# Checking for duplicate values

liqdb_data_cleaned_v1 = liqdb_data_cleaned_v1.drop_duplicates(ignore_index=True)
len(liqdb_data_cleaned_v1)

1472

In [77]:
# Examining total number of healthy and cancer samples

liqdb_data_cleaned_v1["Healthy"].value_counts()

True     1002
False     470
Name: Healthy, dtype: int64

In [78]:
# Performing the label encoding of the target variable

liqdb_data_cleaned_v1['Healthy']=to_categorical(liqdb_data_cleaned_v1['Healthy'],num_classes = 2)

In [82]:
# Saving the processed data

liqdb_data_cleaned_v1.to_csv("cleaned_data/processed_data_v1.csv",index = False)